In [6]:
from tree_sitter import Language, Parser
from tree_sitter_typescript import language_tsx, language_typescript

tsx_language = Language(language_tsx())
tsx_parser = Parser(tsx_language)

typescript_language = Language(language_typescript())
typescript_parser = Parser(typescript_language)

In [23]:
from pathlib import Path
import os
import re

In [10]:
SRC_DIR = Path("../data/app/src").resolve()


In [11]:

def _should_ignore(path):
    IGNORE_DIRS = {
        "src/types",
        "src/styles",
        "src/stories",
        "src/setup",
        "src/languages",
        "src/utils",
        "src/libs/API/parameters",
    }
    IGNORE_FILES = {
        "src/libs/DateUtils.ts",
    }

    if any(path.startswith(ignored_dir) for ignored_dir in IGNORE_DIRS):
        return True
    if path in IGNORE_FILES:
        return True
    return False


In [28]:
FUNCTION_SIG_QUERY = r"""
; ── ordinary 'function foo(param: T)' ───────────────────────
(
  function_declaration
      name:        (identifier)        @name
      parameters:  (formal_parameters) @params
)

; ── const foo = (param) => …  AND  export const foo … ───────
(
  lexical_declaration
      (variable_declarator
          name: (identifier)           @name
          value:
            (arrow_function
                parameters: (formal_parameters) @params))
)

; ── single-parameter arrow:  const foo = bar => … ───────────
(
  lexical_declaration
      (variable_declarator
          name: (identifier)           @name
          value:
            (arrow_function
                parameter: (identifier) @params))
)

; no need for class methods
"""


JSX_RETURN_QUERY = """
(
  return_statement
    (jsx_element)               @jsx
)
(
  return_statement
    (parenthesized_expression
      (jsx_element)             @jsx)
)
"""

In [29]:
ts_query_funcsig  = typescript_language.query(FUNCTION_SIG_QUERY)
tsx_query_funcsig = tsx_language.query(FUNCTION_SIG_QUERY)

tsx_query_jsx   = tsx_language.query(JSX_RETURN_QUERY)   # jsx only matters in .tsx


In [26]:
summaries = {}

for root, _, files in os.walk(SRC_DIR):
        for fname in files:
            path        = Path(root) / fname
            rel_path    = re.sub(r".*?(src/.*)", r"\1", str(path))
            is_tsx_file = rel_path.endswith(".tsx")

            if _should_ignore(rel_path) or rel_path.endswith("types.ts"):
                continue
            try:
                text = path.read_text(encoding="utf-8", errors="ignore")
            except Exception as e:
                logger.error("Could not read %s: %s", rel_path, e)
                continue
            parser   = tsx_parser if is_tsx_file else typescript_parser
            tree     = parser.parse(text.encode())
            src_bytes = text.encode()


In [18]:
import re
kept_lines = []
keep = False

file_header_re = re.compile(r'^diff --git a/(.*?) b/')
for line in diff_text.splitlines(keepends=True):
    m = file_header_re.match(line)
    if m:
        # reset flag at every new file header
        keep = m.group(1).startswith("src/") and not m.group(1).startswith("src/languages")
        
    if keep:
        kept_lines.append(line)

src_only_diff = "".join(kept_lines)

# print(src_only_diff)

print(len(src_only_diff.split()))
print(len(diff_text.split()))


85649
166498
